# Preencher datas e escolher filial

In [14]:
# Data de faturamento (a partir de)
data_inicial_saida_entrega = "30/06/2026"

# Data a ser exibida no excel do protocolo de entrega
data_no_protocolo = "01/07/2026"

# Escolher filial ao trocar chaves da requisição. Altera nome do arquivo excel gerado
# filial = "distribuidora"
filial = "comercio"

# Imports

In [15]:
import requests
import pandas as pd
import os
import json
from dotenv import load_dotenv
from openpyxl import load_workbook
from openpyxl.drawing.image import Image

In [16]:
# Acumulador dos df_nfs de cada execução (uma por filial). Só inicializa se ainda não existir
# no kernel, para não perder o resultado da filial já rodada ao reexecutar com outra filial.
if "lista_df_nfs" not in globals():
    lista_df_nfs = []

# Request API

In [17]:
url = "https://app.omie.com.br/api/v1/produtos/nfconsultar/"

# Carrega o .env e escolhe credenciais por filial
load_dotenv()
# Procura variáveis específicas por filial, com fallback para variáveis genéricas
if filial.lower() == "distribuidora":
    app_key = os.getenv("APP_KEY_DISTRIBUIDORA") or os.getenv("app_key") or os.getenv("APP_KEY")
    app_secret = os.getenv("APP_SECRET_DISTRIBUIDORA") or os.getenv("app_secret") or os.getenv("APP_SECRET")
else:
    app_key = os.getenv("APP_KEY_COMERCIO") or os.getenv("app_key") or os.getenv("APP_KEY")
    app_secret = os.getenv("APP_SECRET_COMERCIO") or os.getenv("app_secret") or os.getenv("APP_SECRET")
print(f"Usando credenciais para filial: {filial}")

payload = {
    "call": "ListarNF",
    "app_key": app_key,
    "app_secret": app_secret,
    "param": [{
  "pagina": 1,
  "registros_por_pagina": 100,
  "ordenar_por": "CODIGO",
  "dSaiEntInicial": data_inicial_saida_entrega,
  "tpNF": 1
}]
}

response = requests.post(url, json=payload)

if response.status_code != 200:
    print(f"Erro na requisição da página 1")
else:
    data = response.json()

# Extrai os dados das notas fiscais do json e armazena em uma lista de tuplas (cnpj_cpf, numero_nf)
nfs = data.get("nfCadastro", [])
todas_nfs = []
for nf in nfs:
    numero_nf = nf.get('ide', {}).get('nNF').lstrip('0')
    cnpj_cpf = nf.get('nfDestInt', {}).get('cnpj_cpf')
    todas_nfs.append((cnpj_cpf, numero_nf))

print(todas_nfs)

Usando credenciais para filial: comercio
[('34.035.902/0003-47', '9924'), ('34.035.902/0002-66', '9925'), ('10.254.717/0005-47', '9926'), ('10.254.717/0004-66', '9927'), ('10.254.717/0008-90', '9928'), ('02.318.956/0005-95', '9929'), ('02.318.956/0003-23', '9930'), ('02.318.956/0012-14', '9931'), ('02.318.956/0013-03', '9932'), ('02.318.956/0006-76', '9933'), ('34.035.902/0006-90', '9934'), ('34.035.902/0005-09', '9935'), ('34.035.902/0001-85', '9936'), ('10.254.717/0002-02', '9937'), ('02.318.956/0007-57', '9938'), ('02.318.956/0004-04', '9939'), ('02.318.956/0020-24', '9940'), ('12.365.759/0002-38', '9941'), ('12.365.759/0004-08', '9942'), ('02.318.956/0001-61', '9943'), ('02.318.956/0011-33', '9944'), ('12.365.759/0003-19', '9945'), ('12.365.759/0001-57', '9946'), ('04.879.925/0001-05', '9947'), ('45.236.954/0001-36', '9948'), ('45.236.954/0002-17', '9949'), ('02.318.956/0008-38', '9950'), ('02.318.956/0014-86', '9951'), ('02.318.956/0015-67', '9952')]


# Dataframe de Produtos das NFs

In [18]:
# Monta uma linha por produto de cada NF, com os dados solicitados
registros = []
for nf in nfs:
    cnpj_cpf = nf.get('nfDestInt', {}).get('cnpj_cpf')
    nnf = nf.get('ide', {}).get('nNF', '').lstrip('0')
    nid_transp = nf.get('compl', {}).get('nIdTransp')

    for item in nf.get('det', []):
        prod = item.get('prod', {})
        registros.append({
            'cnpj_cpf': cnpj_cpf,
            'xProd': prod.get('xProd'),
            'qCom': prod.get('qCom'),
            'uCom': prod.get('uCom'),
            'nnf': nnf,
            'nIdTransp': nid_transp,
            'filial': filial,
        })

df_nfs = pd.DataFrame(registros)
df_nfs

,cnpj_cpf,xProd,qCom,uCom,nnf,nIdTransp,filial
0,34.035.902/0003-47,BATATA ASTERIX (KG),50,KG,9924,8603320478,comercio
1,34.035.902/0002-66,BATATA ASTERIX (KG),100,KG,9925,8603320478,comercio
2,10.254.717/0005-47,BATATA ASTERIX (KG),250,KG,9926,8603320478,comercio
3,10.254.717/0004-66,BATATA ASTERIX (KG),50,KG,9927,8603320478,comercio
4,10.254.717/0008-90,BATATA ASTERIX (KG),100,KG,9928,8603320478,comercio
5,02.318.956/0005-95,BATATA ESCOVADA FL (KG),25,KG,9929,8603320478,comercio
6,02.318.956/0003-23,BATATA ESCOVADA FL (KG),50,KG,9930,8603320478,comercio
7,02.318.956/0012-14,BATATA ESCOVADA FL (KG),75,KG,9931,8603320478,comercio
8,02.318.956/0013-03,BATATA ESCOVADA FL (KG),50,KG,9932,8603320478,comercio
9,02.318.956/0006-76,BATATA ESCOVADA FL (KG),50,KG,9933,8603320478,comercio


# Nome Fantasia no Dataframe

In [19]:
# Adiciona a coluna de nome fantasia no df_nfs a partir de cnpj_nomefantasia.csv
df_nomes = pd.read_csv("cnpj_nomefantasia.csv")
cnpj_to_fantasia = dict(zip(df_nomes["CNPJ"], df_nomes["nome_fantasia"]))
df_nfs["nome_fantasia"] = df_nfs["cnpj_cpf"].map(cnpj_to_fantasia)
df_nfs

,cnpj_cpf,xProd,qCom,uCom,nnf,nIdTransp,filial,nome_fantasia
0,34.035.902/0003-47,BATATA ASTERIX (KG),50,KG,9924,8603320478,comercio,ASTOR SP
1,34.035.902/0002-66,BATATA ASTERIX (KG),100,KG,9925,8603320478,comercio,ASTOR JK
2,10.254.717/0005-47,BATATA ASTERIX (KG),250,KG,9926,8603320478,comercio,ICI JK
3,10.254.717/0004-66,BATATA ASTERIX (KG),50,KG,9927,8603320478,comercio,ICI VILLA LOBOS
4,10.254.717/0008-90,BATATA ASTERIX (KG),100,KG,9928,8603320478,comercio,ICI ALPHAVILLE
5,02.318.956/0005-95,BATATA ESCOVADA FL (KG),25,KG,9929,8603320478,comercio,PIRAJÁ VILLA LOBOS
6,02.318.956/0003-23,BATATA ESCOVADA FL (KG),50,KG,9930,8603320478,comercio,PIRAJÁ ALPHAVILLE
7,02.318.956/0012-14,BATATA ESCOVADA FL (KG),75,KG,9931,8603320478,comercio,PIRAJÁ PRAINHA ITAIM
8,02.318.956/0013-03,BATATA ESCOVADA FL (KG),50,KG,9932,8603320478,comercio,PIRAJÁ TAMBORE
9,02.318.956/0006-76,BATATA ESCOVADA FL (KG),50,KG,9933,8603320478,comercio,ORIGINAL


# Transportador no Dataframe

In [20]:
# Carrega o mapeamento nIdTransp -> transportador a partir de nIDTransp.txt (um json por linha)
mapa_transportadores = []
with open("nIDTransp.txt", encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if linha:
            mapa_transportadores.append(json.loads(linha))

nid_to_transportador = {m["nIdTransp"]: m["transportador"] for m in mapa_transportadores}

# Substitui nIdTransp pelo nome do transportador correspondente
df_nfs["nIdTransp"] = df_nfs["nIdTransp"].astype(str).map(nid_to_transportador).fillna(df_nfs["nIdTransp"])
df_nfs

,cnpj_cpf,xProd,qCom,uCom,nnf,nIdTransp,filial,nome_fantasia
0,34.035.902/0003-47,BATATA ASTERIX (KG),50,KG,9924,Wesley,comercio,ASTOR SP
1,34.035.902/0002-66,BATATA ASTERIX (KG),100,KG,9925,Wesley,comercio,ASTOR JK
2,10.254.717/0005-47,BATATA ASTERIX (KG),250,KG,9926,Wesley,comercio,ICI JK
3,10.254.717/0004-66,BATATA ASTERIX (KG),50,KG,9927,Wesley,comercio,ICI VILLA LOBOS
4,10.254.717/0008-90,BATATA ASTERIX (KG),100,KG,9928,Wesley,comercio,ICI ALPHAVILLE
5,02.318.956/0005-95,BATATA ESCOVADA FL (KG),25,KG,9929,Wesley,comercio,PIRAJÁ VILLA LOBOS
6,02.318.956/0003-23,BATATA ESCOVADA FL (KG),50,KG,9930,Wesley,comercio,PIRAJÁ ALPHAVILLE
7,02.318.956/0012-14,BATATA ESCOVADA FL (KG),75,KG,9931,Wesley,comercio,PIRAJÁ PRAINHA ITAIM
8,02.318.956/0013-03,BATATA ESCOVADA FL (KG),50,KG,9932,Wesley,comercio,PIRAJÁ TAMBORE
9,02.318.956/0006-76,BATATA ESCOVADA FL (KG),50,KG,9933,Wesley,comercio,ORIGINAL


# Consolidar Dataframes das Filiais

In [21]:
# Substitui no acumulador o resultado anterior desta filial (evita duplicar ao reexecutar a mesma filial)
# e junta com os resultados das demais filiais já rodadas neste kernel
lista_df_nfs = [df for df in lista_df_nfs if not (df["filial"] == filial).all()]
lista_df_nfs.append(df_nfs)

df_nfs_consolidado = pd.concat(lista_df_nfs, ignore_index=True)
df_nfs_consolidado

,cnpj_cpf,xProd,qCom,uCom,nnf,nIdTransp,filial,nome_fantasia
0,57.462.972/0001-15,BATATA ESCOVADA FL (KG),25,KG,42617,Luan,distribuidora,CEPA BAR
1,34.791.807/0001-01,BATATA ESCOVADA FL (KG),25,KG,42618,Luan,distribuidora,GINGER TUCUNA
2,28.186.438/0001-25,BATATA ESCOVADA FL (KG),25,KG,42619,Luan,distribuidora,GINGER PINHEIROS
3,62.452.992/0001-45,BATATA ASTERIX (KG),400,KG,42620,Wesley,distribuidora,ZDELI ITAIM
4,62.251.536/0001-37,BATATA ASTERIX (KG),400,KG,42621,Jaylton,distribuidora,ZDELI REBOUÇAS
5,58.558.570/0001-81,BATATA ASTERIX (KG),250,KG,42622,Jaylton,distribuidora,ZDELI HADDOCK LOBO
6,18.786.642/0004-76,BATATA ASTERIX (KG),275,KG,42623,Jaylton,distribuidora,ZDELI IAB - REPUBLICA
7,43.986.357/0001-01,BATATA ASTERIX (KG),75,KG,42624,Jaylton,distribuidora,A5HL (ADEGA A5HL)
8,15.043.489/0001-56,BATATA ASTERIX (KG),100,KG,42625,Jaylton,distribuidora,BBM (Adega Shopping)
9,03.572.023/0001-69,BATATA ASTERIX (KG),75,KG,42626,Luan,distribuidora,SANTE BAR (ADEGA BAR)


# Sacos de Batata

In [22]:
# Cria sacos_batata: para produtos com "batata" no nome, qCom convertido em sacos de 25kg
eh_batata = df_nfs_consolidado["xProd"].str.contains("batata", case=False, na=False)
df_nfs_consolidado["sacos_batata"] = None
df_nfs_consolidado.loc[eh_batata, "sacos_batata"] = df_nfs_consolidado.loc[eh_batata, "qCom"] / 25
df_nfs_consolidado

,cnpj_cpf,xProd,qCom,uCom,nnf,nIdTransp,filial,nome_fantasia,sacos_batata
0,57.462.972/0001-15,BATATA ESCOVADA FL (KG),25,KG,42617,Luan,distribuidora,CEPA BAR,1.0
1,34.791.807/0001-01,BATATA ESCOVADA FL (KG),25,KG,42618,Luan,distribuidora,GINGER TUCUNA,1.0
2,28.186.438/0001-25,BATATA ESCOVADA FL (KG),25,KG,42619,Luan,distribuidora,GINGER PINHEIROS,1.0
3,62.452.992/0001-45,BATATA ASTERIX (KG),400,KG,42620,Wesley,distribuidora,ZDELI ITAIM,16.0
4,62.251.536/0001-37,BATATA ASTERIX (KG),400,KG,42621,Jaylton,distribuidora,ZDELI REBOUÇAS,16.0
5,58.558.570/0001-81,BATATA ASTERIX (KG),250,KG,42622,Jaylton,distribuidora,ZDELI HADDOCK LOBO,10.0
6,18.786.642/0004-76,BATATA ASTERIX (KG),275,KG,42623,Jaylton,distribuidora,ZDELI IAB - REPUBLICA,11.0
7,43.986.357/0001-01,BATATA ASTERIX (KG),75,KG,42624,Jaylton,distribuidora,A5HL (ADEGA A5HL),3.0
8,15.043.489/0001-56,BATATA ASTERIX (KG),100,KG,42625,Jaylton,distribuidora,BBM (Adega Shopping),4.0
9,03.572.023/0001-69,BATATA ASTERIX (KG),75,KG,42626,Luan,distribuidora,SANTE BAR (ADEGA BAR),3.0


# Sacos de Batata por Transportador

In [23]:
# Soma de sacos_batata por xProd e nIdTransp (apenas produtos de batata) e exporta
df_sacos_por_transportador = (
    df_nfs_consolidado[df_nfs_consolidado["sacos_batata"].notna()]
    .groupby(["nIdTransp", "xProd"], as_index=False)["sacos_batata"]
    .sum()
)

df_sacos_por_transportador.to_excel("output/sacos_por_transportador.xlsx", index=False)
df_sacos_por_transportador

,nIdTransp,xProd,sacos_batata
0,Jaylton,BATATA ASTERIX (KG),73.0
1,Jaylton,BATATA ESCOVADA FL (KG),27.0
2,Jaylton,BATATA ESCOVADA NV (KG),18.0
3,Luan,BATATA ASTERIX (KG),5.4
4,Luan,BATATA ESCOVADA FL (KG),14.0
5,Wesley,BATATA ASTERIX (KG),39.0
6,Wesley,BATATA ESCOVADA FL (KG),17.0


# Exportar Protocolo Consolidado

In [24]:
# Exporta df_nfs_consolidado apenas com as colunas solicitadas, na ordem pedida
colunas_exportacao = ["nIdTransp", "nome_fantasia", "xProd", "sacos_batata"]
df_nfs_consolidado[colunas_exportacao].to_excel("output/protocolo_nfs_consolidado.xlsx", index=False)

# Algoritmo Nome Fantasia

In [25]:
#importa "clientes_concatenados_atualizada.csv" para variável df_concatenated
df_concatenated = pd.read_csv("clientes_concatenados_atualizada.csv")

#cria cnpj_col para verificar qual coluna de CNPJ existe
cnpj_col = 'cnpj_cpf' if 'cnpj_cpf' in df_concatenated.columns else 'CNPJ'

# substitui os CNPJs de todas_nfs pelos nomes fantasia usando o df_concatenated atualizado
nome_fantasia_col = 'nome_fantasia' if 'nome_fantasia' in df_concatenated.columns else 'Nome Fantasia'
cnpj_to_nome = dict(zip(df_concatenated[cnpj_col], df_concatenated[nome_fantasia_col]))
todas_nfs = [(cnpj_to_nome.get(cnpj, cnpj), num) for cnpj, num in todas_nfs]

print(f"Substituição com df_concatenated atualizado concluída. Total de NFs: {len(todas_nfs)}")
print(todas_nfs)


Substituição com df_concatenated atualizado concluída. Total de NFs: 29
[('ASTOR SP', '9924'), ('ASTOR JK', '9925'), ('ICI JK', '9926'), ('ICI VL LOBOS', '9927'), ('ICI ALPHAVILLE', '9928'), ('PIRA VL LOBOS', '9929'), ('PIRA ALPHAVILLE', '9930'), ('PRAINHA ITAIM', '9931'), ('PIRA TAMBORE', '9932'), ('ORIGINAL', '9933'), ('ASTOR VL NOVA CONCEICAO', '9934'), ('ASTOR CETENCO', '9935'), ('ASTOR OF', '9936'), ('ICI BELA CINTRA', '9937'), ('PIRA PAULISTA', '9938'), ('PIRA MORUMBI', '9939'), ('PIRA CIDADE SP', '9940'), ('LC2', '9941'), ('LC4', '9942'), ('PIRA PINHEIRO', '9943'), ('PIRA PRAINHA', '9944'), ('LC3', '9945'), ('LC6', '9946'), ('ICI BISTRÔ', '9947'), ('IXE COTOXÓ', '9948'), ('IXE CARAÍBAS', '9949'), ('PIRA ELDORADO', '9950'), ('PIRA CENTER NORTE', '9951'), ('PIRA VL MARIANA', '9952')]


# Algoritmo Planilha Protocolo

In [26]:
# Configuração de blocos
blocos_por_planilha = 5
linhas_por_bloco = 13
linha_inicio = 7

workbook_index = 1
block_index = 0

wb = load_workbook("protocolo_entrega_alterado.xlsx")
ws = wb.active

for i in range(0, len(todas_nfs), 2):
    # Inicia um novo workbook a cada blocos_por_planilha blocos
    if block_index >= blocos_por_planilha:
        file_name = f"{filial}_protocolo_entrega_parte_{workbook_index}.xlsx"

        celulas = ['A1', 'A14', 'A27', 'A40', 'A53', 'I1', 'I14', 'I27', 'I40', 'I53']
        for celula in celulas:
            nova_img = Image('logo_v2.png')
            nova_img.width = 110
            nova_img.height = 95
            ws.add_image(nova_img, celula)

        wb.save(f"C:\\projetos\\protocolo_nfs_projetos\\output\\{file_name}")

        print(f"Salvou {file_name} com {block_index} blocos")

        workbook_index += 1
        block_index = 0
        wb = load_workbook("protocolo_entrega_alterado.xlsx")
        ws = wb.active

    lote = todas_nfs[i:i+2]
    print(f"Processando lote {block_index+1} da planilha {workbook_index}: {lote}")

    row_base = linha_inicio + block_index * linhas_por_bloco
    print(f"Base row para este bloco: {row_base}")

    if len(lote) == 2:
        cnpj1, num1 = lote[0]
        cnpj2, num2 = lote[1]

        # Escreve CNPJs em pares na mesma linha
        ws.cell(row=row_base, column=3, value=cnpj1)   # Coluna C
        ws.cell(row=row_base, column=11, value=cnpj2)  # Coluna K

        # Escreve data_no_protocolo em pares na linha seguinte
        ws.cell(row=row_base + 1, column=4, value=data_no_protocolo)   # Coluna C
        ws.cell(row=row_base + 1, column=12, value=data_no_protocolo)  # Coluna K

        # Escreve números em pares na linha seguinte
        ws.cell(row=row_base + 3, column=4, value=num1)  # Coluna C
        ws.cell(row=row_base + 3, column=12, value=num2) # Coluna K

        print(f"Bloco {block_index+1} planilha {workbook_index}: row {row_base}/{row_base+3} -> {cnpj1},{cnpj2}/{num1},{num2}")

    elif len(lote) == 1:
        cnpj1, num1 = lote[0]

        ws.cell(row=row_base, column=3, value=cnpj1)   # Coluna C
        ws.cell(row=row_base + 1, column=4, value=data_no_protocolo) # Coluna C
        ws.cell(row=row_base + 3, column=3, value=num1) # Coluna C (seguindo o padrão de par)
        
       

        print(f"Bloco {block_index+1} planilha {workbook_index} (solitário): row {row_base}/{row_base+3} -> {cnpj1}/{num1}")

    else:
        continue

    block_index += 1

# Salva o último workbook em aberto
if block_index > 0:
    file_name = f"{filial}_protocolo_entrega_parte_{workbook_index}.xlsx"
    
    celulas = ['A1', 'A14', 'A27', 'A40', 'A53', 'I1', 'I14', 'I27', 'I40', 'I53']
    for celula in celulas:
        nova_img = Image('logo_v2.png')
        nova_img.width = 110
        nova_img.height = 95
        ws.add_image(nova_img, celula)

    wb.save(f"C:\\projetos\\protocolo_nfs_projetos\\output\\{file_name}")
    print(f"Salvou {file_name} com {block_index} blocos")

Processando lote 1 da planilha 1: [('ASTOR SP', '9924'), ('ASTOR JK', '9925')]
Base row para este bloco: 7
Bloco 1 planilha 1: row 7/10 -> ASTOR SP,ASTOR JK/9924,9925
Processando lote 2 da planilha 1: [('ICI JK', '9926'), ('ICI VL LOBOS', '9927')]
Base row para este bloco: 20
Bloco 2 planilha 1: row 20/23 -> ICI JK,ICI VL LOBOS/9926,9927
Processando lote 3 da planilha 1: [('ICI ALPHAVILLE', '9928'), ('PIRA VL LOBOS', '9929')]
Base row para este bloco: 33
Bloco 3 planilha 1: row 33/36 -> ICI ALPHAVILLE,PIRA VL LOBOS/9928,9929
Processando lote 4 da planilha 1: [('PIRA ALPHAVILLE', '9930'), ('PRAINHA ITAIM', '9931')]
Base row para este bloco: 46
Bloco 4 planilha 1: row 46/49 -> PIRA ALPHAVILLE,PRAINHA ITAIM/9930,9931
Processando lote 5 da planilha 1: [('PIRA TAMBORE', '9932'), ('ORIGINAL', '9933')]
Base row para este bloco: 59
Bloco 5 planilha 1: row 59/62 -> PIRA TAMBORE,ORIGINAL/9932,9933
Salvou comercio_protocolo_entrega_parte_1.xlsx com 5 blocos
Processando lote 1 da planilha 2: [('AS